<a href="https://colab.research.google.com/github/paolobalasso/DataScienceCourse/blob/main/notebooks/01_churn_cost_threshold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 01 · Churn — Beyond the Threshold

**Data Science · Università Cattolica del Sacro Cuore, Milano**
Instructor: *Paolo Balasso*

---

## What this notebook adds to the slides

Slides 36–49 of the Main deck already covered:
- Logistic regression for churn (formula, β interpretation, MLE)
- Feature engineering (behavioural, contractual, geographic)
- Confusion matrix → Accuracy / Precision / Recall / F1
- **Cost matrix and the optimal-threshold formula** τ* = C_FP / (C_FP + C_FN)
- Worked numeric examples and hands-on at τ = 0.5 then τ*

This notebook fills **four gaps** that did not fit on slides:

| # | Topic | Why it matters |
|---|-------|----------------|
| **G1** | **ROC vs PR curve** | ROC looks great under class imbalance even when the model is useless. PR tells you the truth. |
| **G2** | **Calibration** (Platt / isotonic + reliability diagram + Brier) | A model can rank perfectly and still output mis-calibrated probabilities. Cost-based decisions need *calibrated* probabilities. |
| **G3** | **Lift & Gain curves + deciles** | The marketing team thinks in *who do we call first*, not in *AUC*. |
| **G4** | **Propensity binning into action tiers** + cenno uplift modelling | From a probability to a business action: VIP retention call, email, no contact. |

**Style.** Short, readable functions. Mini-exercises are mostly *parameter tweaks* — change a cost, swap a calibrator, regroup into quintiles — to reinforce the concept, not to test coding skill.

---

## 0 · Setup

In [ ]:
# Colab-friendly install
%pip install -q scikit-learn matplotlib seaborn pandas numpy

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_curve, precision_recall_curve, roc_auc_score, average_precision_score,
    brier_score_loss, confusion_matrix, classification_report,
)

NAVY, GOLD, INK, MUTED = '#1B263B', '#D4A017', '#0F141E', '#556070'
GREEN, RED = '#2D6E3E', '#B32D2D'
sns.set_style('whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK})
np.random.seed(42)
print('Setup complete.')

## 1 · Build a Telco-style synthetic churn dataset

We generate a **synthetic** dataset rather than relying on an external download — so the notebook runs offline and the data-generating process is *transparent*.

The features mirror the Telco-Customer-Churn benchmark structure: tenure, monthly charges, contract type, plus a noisy "competitor coverage gap" feature.

**Class imbalance** is set to ~22% churners (realistic for telecom) — this matters in Section 3 (ROC vs PR).

In [ ]:
def make_churn(n=8000, churn_rate=0.22, seed=42):
    rng = np.random.default_rng(seed)
    # Latent risk score: tenure -, monthly_charges +, contract long -, competitor_gap +
    tenure         = rng.integers(1, 73, n)                  # months 1..72
    monthly_charge = rng.normal(70, 30, n).clip(20, 200)
    contract       = rng.choice([0, 1, 2], n, p=[0.55, 0.25, 0.20])  # 0=monthly,1=1yr,2=2yr
    competitor_gap = rng.normal(0, 1, n)                     # standardised
    payment_auto   = rng.choice([0, 1], n, p=[0.4, 0.6])     # 1 = auto, lower churn
    support_calls  = rng.poisson(1.5, n)
    # Latent risk
    z = (-0.05*tenure + 0.02*monthly_charge - 0.6*contract
         + 0.45*competitor_gap - 0.35*payment_auto + 0.15*support_calls)
    # Calibrate to target churn rate
    z = z - np.quantile(z, 1 - churn_rate)
    p = 1 / (1 + np.exp(-z))
    y = (rng.uniform(size=n) < p).astype(int)
    df = pd.DataFrame({
        'tenure_months':    tenure,
        'monthly_charge':   monthly_charge.round(2),
        'contract_type':    contract,              # 0/1/2
        'competitor_gap':   competitor_gap.round(3),
        'payment_auto':     payment_auto,
        'support_calls':    support_calls,
        'churned':          y,
    })
    return df

df = make_churn()
print(f'Shape: {df.shape}')
print(f'Churn rate: {df["churned"].mean():.2%}')
df.head()

In [ ]:
# Quick EDA: feature distributions by churn label
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
plots = [
    ('tenure_months',   'Tenure (months)'),
    ('monthly_charge',  'Monthly charge (€)'),
    ('contract_type',   'Contract type (0=monthly, 2=2yr)'),
    ('competitor_gap',  'Competitor coverage gap (z)'),
    ('payment_auto',    'Auto-payment'),
    ('support_calls',   'Support calls (last 90d)'),
]
for ax, (col, label) in zip(axes.flat, plots):
    for c, color, name in [(0, NAVY, 'Stayed'), (1, RED, 'Churned')]:
        ax.hist(df.loc[df['churned']==c, col], bins=25, alpha=0.55,
                color=color, label=name, edgecolor='white', linewidth=0.3)
    ax.set_title(label, fontweight='bold', color=INK, fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('Feature distributions by churn label',
             fontweight='bold', color=INK, y=1.02)
plt.tight_layout(); plt.show()

## 2 · Fit two models (recap — slides 36–47 already explained logistic)

We fit **Logistic Regression** (the slide model) and **Random Forest** (a non-linear comparator). We only need their `predict_proba` outputs for everything that follows.

In [ ]:
X = df.drop(columns='churned')
y = df['churned'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30,
                                          stratify=y, random_state=42)

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr); X_te_s = scaler.transform(X_te)

logreg = LogisticRegression(max_iter=1000).fit(X_tr_s, y_tr)
rf     = RandomForestClassifier(n_estimators=300, max_depth=8,
                                random_state=42, n_jobs=-1).fit(X_tr, y_tr)

p_log = logreg.predict_proba(X_te_s)[:, 1]
p_rf  = rf.predict_proba(X_te)[:, 1]

print(f'Test set:    {len(y_te)} customers, {y_te.mean():.2%} churned')
print(f'LogReg AUC:  {roc_auc_score(y_te, p_log):.3f}')
print(f'RF AUC:      {roc_auc_score(y_te, p_rf):.3f}')

## 3 · Gap G1 — ROC vs PR curve under class imbalance

**ROC** plots TPR vs FPR. With 78% non-churners, the FPR denominator is huge → ROC looks rosy even when precision is poor.

**PR (Precision-Recall)** plots Precision vs Recall. The Precision denominator is the *predicted positives* — a much smaller number — so PR is **far more honest** on imbalanced data.

Rule of thumb in industry: when the positive class is < 30%, always look at PR alongside ROC.

In [ ]:
def plot_roc_pr(y_true, probs_dict):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for name, (p, color) in probs_dict.items():
        # ROC
        fpr, tpr, _ = roc_curve(y_true, p)
        auc = roc_auc_score(y_true, p)
        axes[0].plot(fpr, tpr, color=color, lw=1.6, label=f'{name}  AUC={auc:.3f}')
        # PR
        prec, rec, _ = precision_recall_curve(y_true, p)
        ap = average_precision_score(y_true, p)
        axes[1].plot(rec, prec, color=color, lw=1.6, label=f'{name}  AP={ap:.3f}')
    axes[0].plot([0,1],[0,1], ls='--', color=MUTED, lw=0.8)
    axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC curve', fontweight='bold', color=INK)
    axes[0].legend(loc='lower right')

    baseline = y_true.mean()
    axes[1].axhline(baseline, ls='--', color=MUTED, lw=0.8,
                     label=f'Random ({baseline:.0%})')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall curve', fontweight='bold', color=INK)
    axes[1].legend(loc='lower left')
    plt.tight_layout(); plt.show()

plot_roc_pr(y_te, {
    'LogReg': (p_log, NAVY),
    'Random Forest': (p_rf, GOLD),
})

**Read-out.** Both models look similar on ROC. On PR you can already see which one *finds the churners early*. The PR baseline is the prevalence (~22%) — anything above is doing real work.

> **Mini-exercise (parametric, no coding).** Re-run `make_churn(n=8000, churn_rate=0.05)` — that is a far more imbalanced case (5%). Re-fit and re-plot. Watch how the ROC curves stay nearly identical while the PR curves collapse — *that* is the bias ROC hides.

## 4 · Gap G2 — Calibration: probabilities you can trust

A model can **rank perfectly** (high AUC) and still output probabilities that are wrong as *probabilities*. Logistic regression is usually well-calibrated by construction; tree ensembles like Random Forest are **systematically over-confident at the extremes**.

Why this matters: the cost-based threshold formula τ\* = C_FP / (C_FP + C_FN) assumes the probability is a *real* probability. If your model says 0.8 but the actual frequency at that score is 0.5, you over-act.

Two standard fixes:
- **Platt scaling** (sigmoid): fits a 1-param logistic on the scores
- **Isotonic regression**: non-parametric monotone fit, more flexible but data-hungry

Metric: **Brier score** = mean squared error between predicted prob and outcome (lower = better).

In [ ]:
# Calibrate the RF using both methods (5-fold CV)
rf_platt    = CalibratedClassifierCV(RandomForestClassifier(n_estimators=300, max_depth=8,
                                                             random_state=42, n_jobs=-1),
                                      method='sigmoid', cv=5).fit(X_tr, y_tr)
rf_isotonic = CalibratedClassifierCV(RandomForestClassifier(n_estimators=300, max_depth=8,
                                                             random_state=42, n_jobs=-1),
                                      method='isotonic', cv=5).fit(X_tr, y_tr)

p_rf_pl = rf_platt.predict_proba(X_te)[:, 1]
p_rf_is = rf_isotonic.predict_proba(X_te)[:, 1]

scores = pd.DataFrame({
    'AUC':   [roc_auc_score(y_te, p) for p in [p_log, p_rf, p_rf_pl, p_rf_is]],
    'Brier': [brier_score_loss(y_te, p) for p in [p_log, p_rf, p_rf_pl, p_rf_is]],
}, index=['LogReg (already calibrated)',
          'RF (raw)', 'RF + Platt', 'RF + Isotonic']).round(4)
scores

In [ ]:
# Reliability diagram: predicted probability bins vs observed frequency
fig, ax = plt.subplots(figsize=(7, 6))
for name, p, color in [
    ('LogReg',         p_log,   NAVY),
    ('RF raw',         p_rf,    RED),
    ('RF + Platt',     p_rf_pl, GOLD),
    ('RF + Isotonic',  p_rf_is, GREEN),
]:
    frac_pos, mean_pred = calibration_curve(y_te, p, n_bins=10, strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker='o', lw=1.5, color=color, label=name)
ax.plot([0,1],[0,1], ls='--', color=MUTED, lw=0.8, label='Perfect calibration')
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Observed frequency')
ax.set_title('Reliability diagram (10 quantile bins)',
             fontweight='bold', color=INK)
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

**Read-out.** The raw RF curve usually bows below the diagonal in the upper range (over-confident on high-risk predictions). Platt/isotonic pull it back to the diagonal. Brier score reflects this — lower is better.

> **Mini-exercise (parametric).** Change `n_bins=10` to `n_bins=5` and to `n_bins=20`. Few bins = smoother but less granular; many bins = noisier. There is no "right" answer — there is a *trade-off* to internalise.

## 5 · Gap G3 — Lift & Gain curves + deciles

Marketing teams do not think "AUC". They think **"if we call the top 10% most-likely-to-churn customers, what fraction of actual churners do we catch?"**

- **Gain curve** answers exactly that: cumulative % of churners captured as you walk down the sorted list.
- **Lift curve** = gain / random-baseline. "Lift = 3 in the top decile" means 3× better than random calling.

Used everywhere in CRM, retention, and direct marketing campaigns.

In [ ]:
def gain_lift_table(y_true, p, n_bins=10):
    """Sort customers by predicted prob (desc), bucket into deciles, compute gain & lift."""
    s = pd.DataFrame({'y': y_true, 'p': p}).sort_values('p', ascending=False).reset_index(drop=True)
    s['decile'] = pd.qcut(s.index, n_bins, labels=False) + 1
    grp = s.groupby('decile').agg(
        n=('y', 'size'),
        churners=('y', 'sum'),
        avg_prob=('p', 'mean'),
    )
    total_churners = grp['churners'].sum()
    grp['cum_churners'] = grp['churners'].cumsum()
    grp['gain']         = grp['cum_churners'] / total_churners
    grp['churn_rate']   = grp['churners'] / grp['n']
    grp['lift']         = grp['churn_rate'] / y_true.mean()
    return grp.round(3)

gl_log = gain_lift_table(y_te, p_log, n_bins=10)
gl_rf  = gain_lift_table(y_te, p_rf_is, n_bins=10)

print('LogReg — gain & lift by decile (decile 1 = top 10% riskiest):')
display(gl_log[['n','churners','avg_prob','churn_rate','gain','lift']])

In [ ]:
# Plot gain + lift side-by-side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gain
deciles = np.arange(1, 11)
rand_gain = deciles / 10
axes[0].plot(deciles, rand_gain, ls='--', color=MUTED, lw=0.9, label='Random')
axes[0].plot(deciles, gl_log['gain'].values, marker='o', color=NAVY, lw=1.6, label='LogReg')
axes[0].plot(deciles, gl_rf['gain'].values,  marker='s', color=GOLD, lw=1.6, label='RF + Iso')
axes[0].set_xlabel('Decile of population (1 = riskiest 10%)')
axes[0].set_ylabel('Cumulative % churners captured')
axes[0].set_title('Gain curve', fontweight='bold', color=INK)
axes[0].legend(loc='lower right')

# Lift
axes[1].axhline(1.0, ls='--', color=MUTED, lw=0.9, label='Random = 1.0×')
axes[1].plot(deciles, gl_log['lift'].values, marker='o', color=NAVY, lw=1.6, label='LogReg')
axes[1].plot(deciles, gl_rf['lift'].values,  marker='s', color=GOLD, lw=1.6, label='RF + Iso')
axes[1].set_xlabel('Decile of population'); axes[1].set_ylabel('Lift (vs random)')
axes[1].set_title('Lift curve', fontweight='bold', color=INK)
axes[1].legend(loc='upper right')
plt.tight_layout(); plt.show()

print(f'\nTop-decile lift  (LogReg): {gl_log.loc[1, "lift"]:.2f}x')
print(f'Top-decile gain  (LogReg): {gl_log.loc[1, "gain"]:.1%} of churners captured in 10% of customers')

> **Mini-exercise (parametric).** Switch `n_bins=10` (deciles) to `n_bins=5` (quintiles) or `n_bins=20`. Note how the *shape* of the gain curve does not change — only the resolution — and how the top-bucket lift typically increases as bins shrink (concentration effect). This is *not* a model improvement; it is a binning artefact. A frequent trap in client pitches.

## 6 · Gap G4 — Propensity binning into action tiers

From a probability to a business action. We bin customers into **3 tiers** based on propensity, each tied to a different retention play:

| Tier | Propensity range | Action            | Unit cost |
|------|------------------|-------------------|-----------|
| **VIP retention** | top 10%   | Human agent call, custom offer | €100 |
| **Targeted email** | next 20% | Personalised email + 1-month discount | €5 |
| **No contact**     | bottom 70% | Do nothing | €0 |

This is where the data-science output meets the marketing budget. It is also where calibration matters most — bin thresholds are *probability* thresholds, not score thresholds.

In [ ]:
def propensity_tiers(p, thresholds=(0.70, 0.90)):
    """Map probabilities to tiers based on POPULATION QUANTILES.
       thresholds: (q_low, q_high) → no-contact / email / VIP"""
    q_low  = np.quantile(p, thresholds[0])
    q_high = np.quantile(p, thresholds[1])
    tier = np.where(p >= q_high, 'VIP',
            np.where(p >= q_low,  'Email', 'NoContact'))
    return tier, q_low, q_high

tier, q_low, q_high = propensity_tiers(p_rf_is, thresholds=(0.70, 0.90))
print(f'Quantile cuts on calibrated probabilities: low={q_low:.3f}, high={q_high:.3f}')

tier_summary = pd.DataFrame({'p': p_rf_is, 'y': y_te, 'tier': tier}) \
                .groupby('tier').agg(
                    n=('p', 'size'),
                    avg_p=('p', 'mean'),
                    churn_rate=('y', 'mean'),
                    churners_caught=('y', 'sum'),
                ).reindex(['VIP','Email','NoContact']).round(3)
tier_summary

In [ ]:
# Business simulation: net economic value of the campaign
unit_cost   = {'VIP': 100.0, 'Email': 5.0, 'NoContact': 0.0}
save_rate   = {'VIP': 0.30,  'Email': 0.10, 'NoContact': 0.0}   # share of would-churners actually saved
clv         = 500.0                                              # customer lifetime value at risk

rows = []
for t in ['VIP', 'Email', 'NoContact']:
    n = int(tier_summary.loc[t, 'n'])
    churners = int(tier_summary.loc[t, 'churners_caught'])
    cost     = n * unit_cost[t]
    saved    = churners * save_rate[t]
    value    = saved * clv
    rows.append({
        'Tier': t, 'N': n, 'Cost': cost,
        'Churners would-be': churners,
        'Saved (estimate)': round(saved, 1),
        'Gross value': round(value, 0),
        'Net value': round(value - cost, 0),
    })
biz = pd.DataFrame(rows)
biz.loc['TOTAL'] = ['—', biz['N'].sum(), biz['Cost'].sum(), '', '',
                     biz['Gross value'].sum(), biz['Net value'].sum()]
biz

> **Mini-exercise (parametric).** Try `thresholds=(0.60, 0.85)` (more VIPs / fewer NoContact) and `thresholds=(0.80, 0.95)` (fewer VIPs). The Net Value table will swing dramatically — which threshold pair maximises Net Value?

> **Note.** The save rates (0.30 / 0.10) are educated guesses. In practice they come from an **A/B test or uplift model** (next section).

## 7 · Recap from slides — cost-based threshold (parametric exercise)

The slides derived τ\* = C_FP / (C_FP + C_FN). Let us turn the knob.

- C_FP = cost of contacting a non-churner (wasted offer + brand fatigue)
- C_FN = cost of missing a true churner (lost CLV)

In [ ]:
def expected_cost(p, y, tau, c_fp=30, c_fn=500):
    """Expected cost on the test set given a threshold tau and a cost pair."""
    pred = (p >= tau).astype(int)
    fp = ((pred == 1) & (y == 0)).sum()
    fn = ((pred == 0) & (y == 1)).sum()
    return c_fp * fp + c_fn * fn, fp, fn

# Sweep tau and find empirical optimum vs closed-form
taus = np.linspace(0.01, 0.99, 99)
c_fp_grid = [(30, 500), (50, 500), (100, 500), (30, 1000)]

fig, ax = plt.subplots(figsize=(11, 5))
for c_fp, c_fn in c_fp_grid:
    costs = [expected_cost(p_rf_is, y_te, t, c_fp, c_fn)[0] for t in taus]
    tau_star_closed = c_fp / (c_fp + c_fn)
    tau_star_emp    = taus[int(np.argmin(costs))]
    ax.plot(taus, costs, lw=1.4,
            label=f'C_FP={c_fp}, C_FN={c_fn} → τ*_closed={tau_star_closed:.3f}, τ*_emp={tau_star_emp:.2f}')
ax.set_xlabel('Threshold τ'); ax.set_ylabel('Expected cost (€)')
ax.set_title('Cost-vs-threshold sweep across cost pairs',
             fontweight='bold', color=INK)
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()

**Why empirical τ\* differs slightly from the closed-form.** The closed-form τ\* = C_FP/(C_FP+C_FN) assumes the model's probabilities are perfectly calibrated *and* that we have infinite test data. In a finite sample the empirical minimum is noisy. Once you calibrate (Section 4), the gap shrinks materially.

> **Mini-exercise (parametric).** Add `(20, 800)` to `c_fp_grid` and re-plot. Then change the model to `p_rf` (un-calibrated) and observe how the empirical τ\* drifts further from the closed form.

## 8 · Cenno — Uplift modelling (one paragraph)

The pipeline so far predicts *who will churn*. It does **not** predict *who will be saved by a retention offer*. Those are different populations:

- **Persuadable** — would churn without offer, stays with offer → contact them
- **Sure thing** — stays anyway → wasted contact
- **Lost cause** — churns anyway → wasted contact
- **Do-not-disturb** — would stay, but contacting them actually annoys them into churning → avoid!

**Uplift modelling** estimates the *incremental* effect of the offer, not the absolute probability. Needs an experimental control group. Tools: Two-Model approach, Class Transformation, Causal Forests.

This is the natural next step *after* a calibrated churn model — out of scope for today, but the right mental model for production retention systems.

References: Radcliffe & Surry (2011); Athey & Imbens (2016, *causalforest*).

## 9 · Caveats & next steps

1. **Synthetic data optimism.** Real telco data has missing values, label noise, leakage, drift. Always inspect.
2. **Calibration drifts.** A model calibrated on Q1 may decalibrate by Q3. Monitor Brier score in production.
3. **Top-decile lift overstated** when the test set is too small. Always block-bootstrap confidence intervals.
4. **Cost matrix is not constant.** C_FN (lost CLV) depends on customer segment; a single τ\* across segments is sub-optimal.
5. **Uplift > propensity** for the retention question. Build A/B test infrastructure first.

### Where to go next

- Re-run with `class_weight='balanced'` and compare PR curves
- Replace LogReg with `XGBoost` and re-calibrate
- Bootstrap top-decile lift to put error bars on it
- Build a segment-aware threshold (one τ\* per contract type)

### References

- Niculescu-Mizil & Caruana (2005). *Predicting good probabilities with supervised learning*. ICML.
- Platt (1999). *Probabilistic outputs for support vector machines*.
- Hand & Till (2001). *A simple generalisation of the AUC for multi-class classification*.
- Radcliffe, N. & Surry, P. (2011). *Real-world uplift modelling with significance-based uplift trees*.
- Slides Main FIX8 §6 (slides 36–50) — Churn analytics + cost-based threshold derivation.

---

**End of notebook.** ~10 seconds of runtime on Colab CPU. The goal was to close the four gaps the slides could not cover, with parametric mini-exercises to reinforce the concepts.